# Etapa 2 - Gerar traces

Uma **trace** é a conversa completa que mostra o uso correto (ou não) de uma tool: o prompt de sistema com as ferramentas disponíveis, a pergunta do usuário, a chamada de tool que o modelo faz (quando precisa), um resultado fake dessa tool, e a resposta final do modelo. É esse conjunto de conversas que vai virar o dado de treino do fine-tuning.

### 1. Imports e conexão com a API

In [3]:
from groq import Groq
from dotenv import load_dotenv
import os
import json
import pandas as pd

load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

### 2. Carregar as queries geradas na etapa 1

In [4]:
df_queries = pd.read_csv('queries_geradas.csv')
print(len(df_queries))
df_queries.head()

518


,texto_query,nome_tool,tipo_tool,funcao_tool,dificuldade_query
0,Quero agendar uma reunião para amanhã às 10h c...,schedule_meeting,agenda,Agenda uma reunião com os participantes especi...,facil
1,Preciso marcar uma reunião com todos os funcio...,schedule_meeting,agenda,Agenda uma reunião com os participantes especi...,facil
2,Quero agendar uma reunião com o título 'Reuniã...,schedule_meeting,agenda,Agenda uma reunião com os participantes especi...,facil
3,Gostaria de marcar uma reunião com o Pedro e a...,schedule_meeting,agenda,Agenda uma reunião com os participantes especi...,facil
4,Quero agendar uma reunião para o próximo mês c...,schedule_meeting,agenda,Agenda uma reunião com os participantes especi...,facil


### 3. Carregar o catálogo de tools

In [5]:
with open('tools_catalogo.json', 'r', encoding='utf-8') as f:
    tools_catalogo = json.load(f)

indice_tools = {tool['nome']: tool for tool in tools_catalogo}
print(len(indice_tools))

30


### 4. Função: montar_system_prompt — lista as tools disponíveis

In [6]:
def montar_system_prompt(tool_correta, tools_aleatorias): 
    tools_show = list(tools_aleatorias)
    if tool_correta is not None:
        tools_show.append(tool_correta)

    tools_disponiveis = ""
    for valores in tools_show:
        tools_disponiveis += f'nome:{valores["nome"]}, descrição: {valores["descricao"]}, parâmetros: {valores["parametros"]}\n'

    return tools_disponiveis

print(montar_system_prompt(indice_tools["get_stock_price"], [indice_tools["send_email"], indice_tools["book_flight"]]))


nome:send_email, descrição: Envia um e-mail para o destinatário especificado., parâmetros: [{'nome': 'to', 'tipo': 'string', 'obrigatorio': True}, {'nome': 'subject', 'tipo': 'string', 'obrigatorio': True}, {'nome': 'body', 'tipo': 'string', 'obrigatorio': True}]
nome:book_flight, descrição: Reserva um voo com as opções especificadas., parâmetros: [{'nome': 'origin', 'tipo': 'string', 'obrigatorio': True}, {'nome': 'destination', 'tipo': 'string', 'obrigatorio': True}, {'nome': 'departure_date', 'tipo': 'string', 'obrigatorio': True}]
nome:get_stock_price, descrição: Retorna o preço atual de uma ação específica., parâmetros: [{'nome': 'symbol', 'tipo': 'string', 'obrigatorio': True}]



### 5. Função: gerar_chamada_tool — a chamada da tool em JSON

In [7]:
def gerar_chamada_tool(tool, query):

    prompt = f"""
    Você é um assistente de IA com acesso à seguinte ferramenta (tool):

    nome: {tool["nome"]}
    descrição: {tool["descricao"]}
    parâmetros: {tool["parametros"]}

    Um usuário fez a seguinte pergunta:
    "{query}"

    Gere a chamada dessa ferramenta que resolve a pergunta do usuário, preenchendo cada parâmetro com um valor plausível com base no que foi perguntado. Se a pergunta não deixar um valor explícito para algum parâmetro obrigatório, invente um valor razoável e coerente com o contexto.

    Retorne a resposta em formato JSON, exatamente assim: {{"nome_tool": "{tool["nome"]}", "argumentos": {{...}}}}, sem nenhum texto fora do JSON.
    """

    resposta = client.chat.completions.create(
        model="qwen/qwen3.6-27b",
        messages=[{"role": "user", "content": prompt}],
        max_completion_tokens=500,
        response_format={"type": "json_object"},
        reasoning_effort="none",
)

    texto_resp = resposta.choices[0].message.content.strip()
    texto_resp = texto_resp.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    return json.loads(texto_resp)
    
linha = df_queries[df_queries["dificuldade_query"] == "facil"].iloc[0]
tool = indice_tools[linha["nome_tool"]]
chamada = gerar_chamada_tool(tool, linha["texto_query"])
print(chamada)

    

{'nome_tool': 'schedule_meeting', 'argumentos': {'title': 'Reunião com João e Maria', 'date': 'tomorrow at 10:00', 'participants': ['João', 'Maria']}}


### 6. Função: gerar_resultado_api_mock — resultado fake da tool

In [8]:
def gerar_resultado_api_mock(tool, chamada):
    
    prompt = f"""
    Você está simulando o resultado de uma chamada de API real, para gerar dados de treinamento.

    A ferramenta chamada foi:
    nome: {tool["nome"]}
    descrição: {tool["descricao"]}

    Os argumentos usados nessa chamada foram: {chamada["argumentos"]}

    Invente um resultado plausível e realista que essa ferramenta retornaria para esses argumentos, como se fosse a resposta de uma API de verdade. Pode ser um texto curto, um número, uma confirmação, ou um pequeno JSON — o que fizer mais sentido para o que essa ferramenta faz.

    Retorne APENAS o resultado da ferramenta, sem explicações extras, sem repetir a pergunta e sem frases como "aqui está o resultado".
    """

    resposta = client.chat.completions.create(
        model="qwen/qwen3.6-27b",
        messages=[{"role": "user", "content": prompt}],
        max_completion_tokens=500,
        response_format={"type": "json_object"},
        reasoning_effort="none",
)    
    texto_resp = resposta.choices[0].message.content.strip()     
    texto_resp = texto_resp.removeprefix("```json").removeprefix("```").removesuffix("```").strip()     
    return (texto_resp)

resultado = gerar_resultado_api_mock(tool, chamada)
print(resultado)

{
  "success": true,
  "meeting_id": "mtg_8f4a2b1c9d",
  "status": "scheduled",
  "details": {
    "title": "Reunião com João e Maria",
    "date": "2023-10-26T10:00:00Z",
    "participants": ["João", "Maria"]
  }
}


### 7. Função: gerar_resposta_final — resposta ao usuário usando o resultado

In [9]:
def gerar_resposta_final(query, chamada, resultado):

    prompt = f"""
    Você é um assistente de IA conversando com um usuário.

    O usuário perguntou: "{query}"

    Para responder, você chamou a ferramenta "{chamada["nome_tool"]}" com os argumentos {chamada["argumentos"]}, e ela retornou o seguinte resultado:
    {resultado}

    Escreva a resposta final que você daria ao usuário, em português, de forma natural e direta, como numa conversa normal. Use as informações do resultado acima para responder à pergunta original.

    Não mencione detalhes técnicos como nome da ferramenta, argumentos, JSON ou qualquer coisa que revele que você chamou uma função — fale como se você mesmo soubesse a informação.

    Retorne APENAS a resposta ao usuário, sem introduções como "aqui está a resposta".
    """
    
    resposta = client.chat.completions.create(
        model="qwen/qwen3.6-27b",
        messages=[{"role": "user", "content": prompt}],
        max_completion_tokens=500,
        reasoning_effort="none",
    )
    texto_resp = resposta.choices[0].message.content.strip()     
    texto_resp = texto_resp.removeprefix("```json").removeprefix("```").removesuffix("```").strip()     
    return (texto_resp)

resposta_final = gerar_resposta_final(linha["texto_query"], chamada, resultado)
print(resposta_final)

Pronto, a reunião com o João e a Maria foi agendada para amanhã às 10h.


### 8. Função: gerar_resposta_semtool — resposta direta, sem tool

In [10]:
def gerar_resposta_semtool(query):
    prompt = f"""
    Você é um assistente de IA conversando com um usuário.

    O usuário perguntou: "{query}"

    Essa pergunta pode ser respondida diretamente, sem precisar usar nenhuma ferramenta ou buscar informação externa. Responda de forma natural, útil e direta, em português, como numa conversa normal.

    Responda de forma objetiva, em 2 a 4 frases. Não escreva textos longos nem listas extensas.

    Retorne APENAS a resposta ao usuário, sem introduções como "aqui está a resposta".
    """

    resposta = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        max_completion_tokens=250,
    )
    texto_resp = resposta.choices[0].message.content.strip()
    texto_resp = texto_resp.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    return (texto_resp)

linha_sem_tool = df_queries[df_queries["dificuldade_query"] == "sem_tool"].iloc[0]
resposta_semtool = gerar_resposta_semtool(linha_sem_tool["texto_query"])
print(len(resposta_semtool), "caracteres")
print(resposta_semtool)


431 caracteres
Um motor de combustão interna funciona através da queima de combustível, como gasolina ou diesel, dentro de uma câmara de combustão, onde o ar e o combustível são misturados e inflamados por uma faísca ou compressão, gerando força que é convertida em movimento rotativo. Essa força é então transmitida ao veículo, permitindo seu movimento. O processo envolve quatro etapas principais: admissão, compressão, combustão e escapamento.


### 9. Funções: escolher_tools_aleatorias e gerar_trace_para_linha

In [11]:
import random

def escolher_tools_aleatorias(nome_tool_correta, n_taleatorias=3):
    candidatos = [tool for nome, tool in indice_tools.items() if nome != nome_tool_correta]
    return random.sample(candidatos, n_taleatorias)

def gerar_trace_para_linha(linha):
    nome_tool_correta = linha['nome_tool']
    tool_correta = indice_tools[nome_tool_correta] if pd.notna(nome_tool_correta) else None
    tools_aleatorias = escolher_tools_aleatorias(nome_tool_correta)
    query = linha["texto_query"]

    system_prompt = montar_system_prompt(tool_correta, tools_aleatorias)
    
    if linha["dificuldade_query"] == "sem_tool":
        resposta = gerar_resposta_semtool(query)
        return [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": query},
            {"role": "assistant", "content": resposta},
        ]
    else:
        chamada = gerar_chamada_tool(tool_correta, query)
        chamada_string = json.dumps(chamada)
        resultado = gerar_resultado_api_mock(tool_correta, chamada)
        resposta_final = gerar_resposta_final(query, chamada, resultado)
        return [
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": query},
                    {"role": "assistant", "content": chamada_string},
                    {"role": "user", "content": resultado},
                    {"role": "assistant", "content": resposta_final},
                ]

print(gerar_trace_para_linha(df_queries.iloc[0]))          # uma com tool
print(gerar_trace_para_linha(linha_sem_tool))      



[{'role': 'system', 'content': "nome:manage_finances, descrição: Gerencia as finanças pessoais com base nos gastos e receitas., parâmetros: [{'nome': 'income', 'tipo': 'number', 'obrigatorio': True}, {'nome': 'expenses', 'tipo': 'array', 'obrigatorio': True}]\nnome:get_weather_alerts, descrição: Retorna alertas de condições climáticas adversas., parâmetros: [{'nome': 'location', 'tipo': 'string', 'obrigatorio': True}]\nnome:send_email, descrição: Envia um e-mail para o destinatário especificado., parâmetros: [{'nome': 'to', 'tipo': 'string', 'obrigatorio': True}, {'nome': 'subject', 'tipo': 'string', 'obrigatorio': True}, {'nome': 'body', 'tipo': 'string', 'obrigatorio': True}]\nnome:schedule_meeting, descrição: Agenda uma reunião com os participantes especificados., parâmetros: [{'nome': 'title', 'tipo': 'string', 'obrigatorio': True}, {'nome': 'date', 'tipo': 'string', 'obrigatorio': True}, {'nome': 'participants', 'tipo': 'array', 'obrigatorio': True}]\n"}, {'role': 'user', 'content

### 10. Loop retomável — gera as traces que ainda faltam

In [ ]:
import json

try:
    with open("traces_progresso.json", "r", encoding="utf-8") as f:
        progresso = json.load(f)
except FileNotFoundError:
    progresso = {}

for i, linha in df_queries.iterrows():
    chave = str(i)

    if progresso.get(chave) is not None:
        continue  # já tem trace de verdade, pula sem gastar API

    try:
        trace = gerar_trace_para_linha(linha)
        progresso[chave] = trace
    except Exception as e:
        progresso[chave] = None
        print(f"Linha {i} falhou: {e}")

    with open("traces_progresso.json", "w", encoding="utf-8") as f:
        json.dump(progresso, f, ensure_ascii=False, indent=2)

prontas = sum(1 for v in progresso.values() if v is not None)
print(f"{prontas} de {len(progresso)} linhas com trace pronta")


In [13]:
import json
import random
import time

ESPERA = 0         # segundos de espera quando a cota enche
MAX_TENTATIVAS = 3    # quantas vezes tentar a mesma linha

# --- 1. carrega o progresso salvo ---
try:
    with open("traces_progresso.json", "r", encoding="utf-8") as f:
        progresso = json.load(f)
except FileNotFoundError:
    progresso = {}

# --- 2. monta a fila: sem_tool primeiro, resto embaralhado ---
faltam = [i for i in df_queries.index if progresso.get(str(i)) is None]
sem_tool = [i for i in faltam if df_queries.loc[i, "dificuldade_query"] == "sem_tool"]
com_tool = [i for i in faltam if df_queries.loc[i, "dificuldade_query"] != "sem_tool"]
random.shuffle(com_tool)
fila = sem_tool + com_tool

print(f"{len(fila)} linhas faltando ({len(sem_tool)} sem_tool vao primeiro)")

# --- 3. processa a fila ---
cota_esgotada = False

for posicao, i in enumerate(fila, start=1):
    chave = str(i)
    linha = df_queries.loc[i]
    erro_final = None

    for tentativa in range(1, MAX_TENTATIVAS + 1):
        try:
            progresso[chave] = gerar_trace_para_linha(linha)
            erro_final = None
            break
        except Exception as e:
            erro_final = e
            if "429" in str(e) and tentativa < MAX_TENTATIVAS:
                print(f"  linha {i}: cota cheia, esperando {ESPERA}s ({tentativa}/{MAX_TENTATIVAS})")
                time.sleep(ESPERA)
            else:
                progresso[chave] = None
                break

    with open("traces_progresso.json", "w", encoding="utf-8") as f:
        json.dump(progresso, f, ensure_ascii=False, indent=2)

    if erro_final is None:
        if posicao % 10 == 0:
            prontas = sum(1 for v in progresso.values() if v is not None)
            print(f"[{posicao}/{len(fila)}] {prontas} prontas no total")
    else:
        print(f"[{posicao}/{len(fila)}] linha {i} falhou: {type(erro_final).__name__}")
        if "429" in str(erro_final):
            cota_esgotada = True
            break

# --- 4. resumo ---
prontas = sum(1 for v in progresso.values() if v is not None)
print(f"\n{prontas} de {len(progresso)} linhas com trace pronta")
if cota_esgotada:
    print("Parei porque a cota acabou. Rode esta celula de novo mais tarde.")


208 linhas faltando (97 sem_tool vao primeiro)
[10/208] 320 prontas no total
[20/208] 330 prontas no total
[30/208] 340 prontas no total
[40/208] 350 prontas no total
[50/208] 360 prontas no total
[60/208] 370 prontas no total
[70/208] 380 prontas no total
[80/208] 390 prontas no total
[90/208] 400 prontas no total
[100/208] 410 prontas no total
[110/208] 420 prontas no total
  linha 291: cota cheia, esperando 0s (1/3)
[120/208] 430 prontas no total
  linha 369: cota cheia, esperando 0s (1/3)
[130/208] 440 prontas no total
[140/208] 450 prontas no total
  linha 294: cota cheia, esperando 0s (1/3)
[150/208] 460 prontas no total
[160/208] 470 prontas no total
[170/208] 480 prontas no total
  linha 380: cota cheia, esperando 0s (1/3)
  linha 380: cota cheia, esperando 0s (2/3)
[175/208] linha 380 falhou: RateLimitError

484 de 518 linhas com trace pronta
Parei porque a cota acabou. Rode esta celula de novo mais tarde.
